In [0]:
import requests
import pandas as pd
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# 1. Hit the free REST API endpoint
api_url = "https://jsonplaceholder.typicode.com/posts"
response = requests.get(api_url, timeout=10)

if response.status_code == 200:
    # 2. Extract the raw JSON array
    raw_json_data = response.json()
    
    # 3. Convert to a local Pandas DataFrame on the driver node
    pdf = pd.DataFrame(raw_json_data)
    
    # 4. Enforce structural Spark data types matching the API specification
    spark_schema = StructType([
        StructField("userId", IntegerType(), True),
        StructField("id", IntegerType(), True),
        StructField("title", StringType(), True),
        StructField("body", StringType(), True)
    ])
    
    # 5. Parallelize the data across the cluster workers
    df = spark.createDataFrame(pdf, schema=spark_schema)
    
    # 6. Define the exact physical destination in your Bronze container
    bronze_api_path = "abfss://bronze@saadbprojectsupply.dfs.core.windows.net/raw_api_posts"
    
    # 7. Write the physical files to Azure storage
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .save(bronze_api_path)
      
    print(f"SUCCESS: Ingested {df.count()} records from API directly to storage.")
else:
    raise Exception(f"API Ingestion failed with HTTP status code: {response.status_code}")